In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
%pwd

'd:\\Text-summarizer\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'd:\\Text-summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from text_summarizer.constants import *
from text_summarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        print(f"Config keys found: {self.config.keys()}")
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])

        def get_model_evaluation_config(self) -> ModelEvaluationConfig:
            config = self.config.model_evaluation
            create_directories([config.root_dir])
            model_evaluation_config = ModelEvaluationConfig(
                root_dir = config.root_dir,
                data_path = config.data_path,
                model_path = config.model_path,
                tokenizer_path = config.tokenizer_path,
                metric_file_name = config.metric_file_name
            )
            return model_evaluation_config

In [28]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
from evaluate import load as load_metric
import torch
import pandas as pd
from tqdm import tqdm

In [37]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config
        


    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """Yield successive batch-sized chunks from list_of_elements."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer,
                                batch_size=16, device="cuda" if torch.cuda.is_available() else "cpu",
                                column_text="article",
                                column_summary="highlights"):
        # Convert generators to lists to avoid consuming them prematurely
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):

            # Tokenize inputs
            inputs = tokenizer(article_batch, max_length=1024, truncation=True,
                            padding="max_length", return_tensors="pt")

            # Generate summaries
            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                                    attention_mask=inputs["attention_mask"].to(device),
                                    length_penalty=0.8, num_beams=8, max_length=128)

            # Decode generated tokens to text
            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                                clean_up_tokenization_spaces=True)
                                for s in summaries]

            # Standardize formatting (optional but common for ROUGE)
            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

            # Add batch to the metric object
            metric.add_batch(predictions=decoded_summaries, references=target_batch)

        # Compute and return final ROUGE scores
        score = metric.compute()
        return score
    
    def evaluate(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        
        rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]
        rouge_matrix = load_metric("rouge") # Now this will work after pip install

        score = self.calculate_metric_on_test_ds(
            dataset=dataset_samsum_pt["test"][0:10],
            metric=rouge_matrix,
            model=model_pegasus,
            tokenizer=tokenizer,
            batch_size=2,
            column_text="dialogue",
            column_summary="summary"
        )
            # FIX: Access scores correctly based on the version of rouge library used
            # If using old datasets.load_metric, use score[rn].mid.fmeasure
            # If using new evaluate.load, use score[rn]
        rouge_dict = {}
        for rn in rouge_names:
            if hasattr(score[rn], 'mid'):
                rouge_dict[rn] = score[rn].mid.fmeasure
            else:
                rouge_dict[rn] = score[rn]

        df = pd.DataFrame(rouge_dict, index=['pegasus'])
        df.to_csv(self.config.metric_file_name, index=False)
        print("Evaluation results saved to:", self.config.metric_file_name)


In [38]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-03-28 23:00:20,687: INFO: common]: yaml file: config\config.yaml loaded successfully
Config keys found: dict_keys(['artifacts_root', 'data_ingestion', 'data_validation', 'data_transformation', 'model_trainer', 'model_evaluation'])
[2026-03-28 23:00:20,691: INFO: common]: yaml file: params.yaml loaded successfully
[2026-03-28 23:00:20,693: INFO: common]: created directory at: artifacts
[2026-03-28 23:00:20,693: INFO: common]: created directory at: artifacts/model_evaluation


100%|██████████| 5/5 [02:50<00:00, 34.07s/it]

[2026-03-28 23:03:19,305: INFO: rouge_scorer]: Using default tokenizer.


Evaluation results saved to: artifacts/model_evaluation/metric.txt


In [14]:
import os

# Option A: Move up one directory to the project root
os.chdir("../") 
model_path = "artifacts/model_trainer/pegasus-samsum-model"

# Option B: Use the full absolute path
model_path = r"d:\Text-summarizer\artifacts\model_trainer\pegasus_samsum_model"

# Load the model
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path)


In [16]:
import os

# Check if we are in the 'research' folder
if os.getcwd().endswith("research"):
    os.chdir("../")  # Move up to the project root

print(f"Current working directory: {os.getcwd()}")


Current working directory: d:\


In [17]:
import os

# Change this to your specific project path
project_path = r"d:\Text-summarizer"
os.chdir(project_path)

print(f"New working directory: {os.getcwd()}")


New working directory: d:\Text-summarizer


In [19]:
import os
path = "artifacts/model_trainer/pegasus_samsum_model"

print(f"Directory exists: {os.path.exists(path)}")
if os.path.exists(path):
    print(f"Files in folder: {os.listdir(path)}")


Directory exists: True
Files in folder: ['config.json', 'generation_config.json', 'model.safetensors']


In [20]:
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# Replace 'checkpoint-XXXX' with your actual folder name (e.g., checkpoint-500)
model_path = "artifacts/model_trainer/checkpoint-51" 

# Load both from the same checkpoint folder
tokenizer = AutoTokenizer.from_pretrained(model_path)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_path)

print("Model and Tokenizer loaded successfully from checkpoint!")


Model and Tokenizer loaded successfully from checkpoint!


In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_pegasus.to(device)


PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [35]:
%pip install nltk rouge_score absl-py


  Using cached nltk-3.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 493.7 kB/s eta 0:00:02
   ------------- -------------------------- 0.5/1.5 MB 493.7 kB/s eta 0:00:02
   ------------- -------------

In [36]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to C:\Users\Tejaswini
[nltk_data]     V\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True